In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
nih_chest_xrays_data_path = kagglehub.dataset_download('nih-chest-xrays/data')

print('Data source import complete.')


In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import gc
import math
import json
import random
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import densenet121, DenseNet121_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

from tqdm.auto import tqdm

In [ ]:
SEED = 42

IMG_SIZE = 224
BATCH_SIZE = 32          # reduce to 16 if OOM
EPOCHS = 8               # first real run
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_DIR = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
CSV_PATH = f"{BASE_DIR}/Data_Entry_2017.csv"
TRAIN_LIST_PATH = f"{BASE_DIR}/train_val_list.txt"
TEST_LIST_PATH = f"{BASE_DIR}/test_list.txt"
BBOX_PATH = f"{BASE_DIR}/BBox_List_2017.csv"

CHECKPOINT_PATH = "/kaggle/working/best_radtriage_model_full.pt"
METRICS_PATH = "/kaggle/working/test_metrics_full.csv"
PREDICTIONS_PATH = "/kaggle/working/test_predictions_full.csv"
WORKLIST_PATH = "/kaggle/working/worklist_simulation_full.csv"
THRESHOLDS_PATH = "/kaggle/working/best_thresholds_full.json"

CLASSES = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural_Thickening",
    "Hernia",
]

print("DEVICE:", DEVICE)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [ ]:
all_pngs = glob(f"{BASE_DIR}/images_*/images/*.png")
print("Total PNG files found:", len(all_pngs))

image_path_map = {os.path.basename(p): p for p in all_pngs}
print("Unique mapped filenames:", len(image_path_map))

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Metadata shape:", df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
df["image_path"] = df["Image Index"].map(image_path_map)

missing_paths = df["image_path"].isna().sum()
print("Missing image paths:", missing_paths)

df = df.dropna(subset=["image_path"]).reset_index(drop=True)
print("Rows after mapping image paths:", len(df))

In [ ]:
def encode_labels(label_str, class_names):
    labels = [x.strip() for x in str(label_str).split("|")]
    target = np.zeros(len(class_names), dtype=np.float32)
    for i, cls in enumerate(class_names):
        if cls in labels:
            target[i] = 1.0
    return target

df["target"] = df["Finding Labels"].apply(lambda x: encode_labels(x, CLASSES))

In [ ]:
all_labels = set()
for s in df["Finding Labels"].astype(str):
    for label in s.split("|"):
        all_labels.add(label.strip())

print(sorted(all_labels))

In [ ]:
targets_full = np.stack(df["target"].values)

label_dist_df = pd.DataFrame({
    "class": CLASSES,
    "positive_count": targets_full.sum(axis=0).astype(int),
    "prevalence": targets_full.sum(axis=0) / len(df)
}).sort_values("positive_count", ascending=False)

label_dist_df

In [ ]:
with open(TRAIN_LIST_PATH, "r") as f:
    train_val_names = [line.strip() for line in f if line.strip()]

with open(TEST_LIST_PATH, "r") as f:
    test_names = [line.strip() for line in f if line.strip()]

print("Train/Val names:", len(train_val_names))
print("Test names:", len(test_names))

In [ ]:
train_val_df = df[df["Image Index"].isin(train_val_names)].reset_index(drop=True)
test_df = df[df["Image Index"].isin(test_names)].reset_index(drop=True)

print("train_val_df:", len(train_val_df))
print("test_df:", len(test_df))

In [ ]:
unique_train_val_patients = train_val_df["Patient ID"].unique()

train_patients, val_patients = train_test_split(
    unique_train_val_patients,
    test_size=0.12,
    random_state=SEED
)

train_df = train_val_df[train_val_df["Patient ID"].isin(train_patients)].reset_index(drop=True)
val_df = train_val_df[train_val_df["Patient ID"].isin(val_patients)].reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

In [ ]:
assert set(train_df["Patient ID"]).isdisjoint(set(val_df["Patient ID"]))
assert set(train_df["Patient ID"]).isdisjoint(set(test_df["Patient ID"]))
assert set(val_df["Patient ID"]).isdisjoint(set(test_df["Patient ID"]))

print("No patient leakage across train/val/test.")

In [ ]:
USE_SMOKE_TEST = False

if USE_SMOKE_TEST:
    train_df_use = train_df.sample(8000, random_state=SEED).reset_index(drop=True)
    val_df_use = val_df.sample(1500, random_state=SEED).reset_index(drop=True)
    test_df_use = test_df.sample(1500, random_state=SEED).reset_index(drop=True)
    epochs_to_run = 2
else:
    train_df_use = train_df
    val_df_use = val_df
    test_df_use = test_df
    epochs_to_run = EPOCHS

print("Train used:", len(train_df_use))
print("Val used:", len(val_df_use))
print("Test used:", len(test_df_use))
print("Epochs:", epochs_to_run)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("L")
        image = Image.merge("RGB", (image, image, image))

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(row["target"], dtype=torch.float32)

        return {
            "image": image,
            "target": target,
            "image_id": row["Image Index"],
            "patient_id": row["Patient ID"],
            "image_path": row["image_path"]
        }

In [ ]:
train_targets = np.stack(train_df_use["target"].values)

pos_counts = train_targets.sum(axis=0)
neg_counts = len(train_targets) - pos_counts
pos_weight = neg_counts / (pos_counts + 1e-6)

class_weight_df = pd.DataFrame({
    "class": CLASSES,
    "pos_count": pos_counts.astype(int),
    "neg_count": neg_counts.astype(int),
    "pos_weight": pos_weight
}).sort_values("pos_count")

class_weight_df

In [ ]:
sample_weights = []
for target in train_targets:
    positive_indices = np.where(target == 1)[0]
    if len(positive_indices) > 0:
        weight = float(np.max(pos_weight[positive_indices]))
    else:
        weight = 1.0
    sample_weights.append(weight)

sample_weights = torch.DoubleTensor(sample_weights)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [ ]:
train_ds = ChestXrayDataset(train_df_use, transform=train_transform)
val_ds = ChestXrayDataset(val_df_use, transform=eval_transform)
test_ds = ChestXrayDataset(test_df_use, transform=eval_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
batch = next(iter(train_loader))
print(batch["image"].shape)
print(batch["target"].shape)
print(batch["image_id"][:3])

In [ ]:
class RadTriageDenseNet(nn.Module):
    def __init__(self, num_classes=14, pretrained=True, dropout=0.2):
        super().__init__()
        weights = DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = densenet121(weights=weights)
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

model = RadTriageDenseNet(num_classes=len(CLASSES)).to(DEVICE)
model

In [ ]:
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight, dtype=torch.float32).to(DEVICE)
)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [ ]:
def compute_multilabel_metrics(y_true, y_prob, thresholds=None):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    if thresholds is None:
        thresholds = np.array([0.5] * y_true.shape[1])

    y_pred = (y_prob >= thresholds).astype(int)

    per_class_auc = {}
    per_class_f1 = {}
    sensitivities = {}
    specificities = {}

    for i, cls in enumerate(CLASSES):
        try:
            auc = roc_auc_score(y_true[:, i], y_prob[:, i])
        except Exception:
            auc = np.nan

        per_class_auc[cls] = auc
        per_class_f1[cls] = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)

        cm = confusion_matrix(y_true[:, i], y_pred[:, i], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        sensitivities[cls] = tp / (tp + fn + 1e-6)
        specificities[cls] = tn / (tn + fp + 1e-6)

    macro_auc = np.nanmean(list(per_class_auc.values()))
    macro_f1 = np.mean(list(per_class_f1.values()))

    return {
        "macro_auc": macro_auc,
        "macro_f1": macro_f1,
        "per_class_auc": per_class_auc,
        "per_class_f1": per_class_f1,
        "sensitivities": sensitivities,
        "specificities": specificities
    }

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0

    for batch in tqdm(loader, leave=False):
        images = batch["image"].to(device)
        targets = batch["target"].to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            images = batch["image"].to(device)
            targets = batch["target"].to(device)

            logits = model(images)
            loss = criterion(logits, targets)

            probs = torch.sigmoid(logits)

            running_loss += loss.item() * images.size(0)
            all_targets.append(targets.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    val_loss = running_loss / len(loader.dataset)
    all_targets = np.concatenate(all_targets, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)

    metrics = compute_multilabel_metrics(all_targets, all_probs)

    return val_loss, metrics, all_targets, all_probs

In [ ]:
best_auc = -np.inf
history = []

for epoch in range(epochs_to_run):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_metrics, val_targets, val_probs = validate(model, val_loader, criterion, DEVICE)

    scheduler.step(val_metrics["macro_auc"])

    print(f"Epoch {epoch+1}/{epochs_to_run}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val Macro AUC: {val_metrics['macro_auc']:.4f}")
    print(f"Val Macro F1:  {val_metrics['macro_f1']:.4f}")

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_auc": val_metrics["macro_auc"],
        "val_macro_f1": val_metrics["macro_f1"]
    })

    if val_metrics["macro_auc"] > best_auc:
        best_auc = val_metrics["macro_auc"]
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print("Saved best model.")

In [ ]:
history_df = pd.DataFrame(history)
history_df

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["val_macro_auc"], label="val_macro_auc")
plt.plot(history_df["epoch"], history_df["val_macro_f1"], label="val_macro_f1")
plt.xlabel("Epoch")
plt.ylabel("Metric")
plt.legend()
plt.show()

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.eval()
print("Loaded best checkpoint from:", CHECKPOINT_PATH)

In [ ]:
_, _, val_targets, val_probs = validate(model, val_loader, criterion, DEVICE)

best_thresholds = []

for i in range(len(CLASSES)):
    if val_targets[:, i].sum() < 5:
        best_thresholds.append(0.5)
        continue

    best_thr = 0.5
    best_f1 = -1

    for thr in np.arange(0.1, 0.91, 0.05):
        preds = (val_probs[:, i] >= thr).astype(int)
        f1 = f1_score(val_targets[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    best_thresholds.append(float(best_thr))

best_thresholds = np.array(best_thresholds)
best_thresholds

In [ ]:
threshold_dict = {cls: float(thr) for cls, thr in zip(CLASSES, best_thresholds)}

with open(THRESHOLDS_PATH, "w") as f:
    json.dump(threshold_dict, f, indent=2)

threshold_dict

In [ ]:
test_loss, test_metrics, test_targets, test_probs = validate(model, test_loader, criterion, DEVICE)
test_metrics_tuned = compute_multilabel_metrics(test_targets, test_probs, thresholds=best_thresholds)

print("Test Loss:", test_loss)
print("Test Macro AUC:", test_metrics_tuned["macro_auc"])
print("Test Macro F1:", test_metrics_tuned["macro_f1"])

In [ ]:
metrics_df = pd.DataFrame({
    "class": CLASSES,
    "auc": [test_metrics_tuned["per_class_auc"][c] for c in CLASSES],
    "f1": [test_metrics_tuned["per_class_f1"][c] for c in CLASSES],
    "sensitivity": [test_metrics_tuned["sensitivities"][c] for c in CLASSES],
    "specificity": [test_metrics_tuned["specificities"][c] for c in CLASSES],
    "threshold": [threshold_dict[c] for c in CLASSES]
})

metrics_df = metrics_df.sort_values("auc", ascending=False).reset_index(drop=True)
metrics_df

In [ ]:
metrics_df.to_csv(METRICS_PATH, index=False)
print("Saved metrics to:", METRICS_PATH)

In [ ]:
pred_df = test_df_use[["Image Index", "Patient ID", "Finding Labels", "image_path"]].copy().reset_index(drop=True)

for i, cls in enumerate(CLASSES):
    pred_df[f"true_{cls}"] = test_targets[:, i].astype(int)
    pred_df[f"prob_{cls}"] = test_probs[:, i]

pred_df.to_csv(PREDICTIONS_PATH, index=False)
print("Saved predictions to:", PREDICTIONS_PATH)
pred_df.head()

In [ ]:
def preprocess_single_image(image_path):
    image = Image.open(image_path).convert("L")
    rgb = Image.merge("RGB", (image, image, image))
    tensor = eval_transform(rgb).unsqueeze(0).to(DEVICE)
    return image, rgb, tensor

In [ ]:
def predict_single_image(model, image_path, thresholds=None):
    model.eval()
    gray_img, rgb_img, tensor = preprocess_single_image(image_path)

    with torch.no_grad():
        logits = model(tensor)
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    if thresholds is None:
        thresholds = np.array([0.5] * len(CLASSES))

    pred_binary = (probs >= thresholds).astype(int)

    result_df = pd.DataFrame({
        "class": CLASSES,
        "probability": probs,
        "predicted": pred_binary
    }).sort_values("probability", ascending=False).reset_index(drop=True)

    return gray_img, rgb_img, tensor, probs, result_df

In [ ]:
sample_path = test_df_use.iloc[0]["image_path"]

gray_img, rgb_img, tensor, probs, result_df = predict_single_image(
    model, sample_path, best_thresholds
)

plt.figure(figsize=(5, 5))
plt.imshow(gray_img, cmap="gray")
plt.axis("off")
plt.show()

result_df.head(10)

In [ ]:
SEVERITY_WEIGHTS = {
    "Atelectasis": 0.35,
    "Cardiomegaly": 0.45,
    "Effusion": 0.55,
    "Infiltration": 0.50,
    "Mass": 0.60,
    "Nodule": 0.30,
    "Pneumonia": 0.75,
    "Pneumothorax": 1.00,
    "Consolidation": 0.65,
    "Edema": 0.80,
    "Emphysema": 0.30,
    "Fibrosis": 0.25,
    "Pleural_Thickening": 0.20,
    "Hernia": 0.15,
}

In [ ]:
def compute_triage_score(probs):
    score = 0.0
    for i, cls in enumerate(CLASSES):
        score += probs[i] * SEVERITY_WEIGHTS[cls]
    return float(score)

def assign_triage_tier(probs, score):
    prob_map = {cls: probs[i] for i, cls in enumerate(CLASSES)}

    if prob_map["Pneumothorax"] >= 0.60:
        return "Emergent"
    elif prob_map["Edema"] >= 0.70 or prob_map["Pneumonia"] >= 0.70:
        return "Urgent"
    elif score >= 0.75:
        return "Emergent"
    elif score >= 0.40:
        return "Urgent"
    else:
        return "Routine"

In [ ]:
score = compute_triage_score(probs)
tier = assign_triage_tier(probs, score)

print("Urgency Score:", score)
print("Triage Tier:", tier)

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        self.fwd_hook = self.target_layer.register_forward_hook(self.save_activation)
        self.bwd_hook = self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, inp, out):
        self.activations = out

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_tensor, class_idx):
        self.model.eval()
        self.model.zero_grad()

        logits = self.model(input_tensor)
        target = logits[:, class_idx]
        target.backward()

        grads = self.gradients[0]
        acts = self.activations[0]

        weights = grads.mean(dim=(1, 2))
        cam = torch.zeros(acts.shape[1:], dtype=torch.float32, device=acts.device)

        for i, w in enumerate(weights):
            cam += w * acts[i]

        cam = torch.relu(cam)
        cam -= cam.min()
        cam /= (cam.max() + 1e-8)

        return cam.detach().cpu().numpy()

    def close(self):
        self.fwd_hook.remove()
        self.bwd_hook.remove()

In [ ]:
def overlay_cam_on_image(gray_image, cam, alpha=0.4):
    gray_np = np.array(gray_image)
    cam_img = Image.fromarray(np.uint8(cam * 255)).resize(gray_image.size)
    cam_np = np.array(cam_img) / 255.0

    plt.figure(figsize=(6, 6))
    plt.imshow(gray_np, cmap="gray")
    plt.imshow(cam_np, cmap="jet", alpha=alpha)
    plt.axis("off")
    plt.show()

In [ ]:
gradcam = GradCAM(model, model.backbone.features.denseblock4)

top_class_idx = int(np.argmax(probs))
print("Top predicted class:", CLASSES[top_class_idx])

cam = gradcam.generate(tensor, top_class_idx)
overlay_cam_on_image(gray_img, cam)

In [ ]:
def run_case(model, image_path, thresholds):
    gray_img, rgb_img, tensor, probs, result_df = predict_single_image(model, image_path, thresholds)
    score = compute_triage_score(probs)
    tier = assign_triage_tier(probs, score)

    top_finding = result_df.iloc[0]["class"]
    top_prob = float(result_df.iloc[0]["probability"])

    return {
        "image_path": image_path,
        "top_finding": top_finding,
        "top_probability": top_prob,
        "urgency_score": score,
        "triage_tier": tier
    }

In [ ]:
sample_cases = test_df_use.sample(10, random_state=SEED).reset_index(drop=True)

results = []
for i, row in sample_cases.iterrows():
    out = run_case(model, row["image_path"], best_thresholds)
    out["original_rank"] = i + 1
    out["image_id"] = row["Image Index"]
    results.append(out)

worklist_df = pd.DataFrame(results)
worklist_df = worklist_df.sort_values("urgency_score", ascending=False).reset_index(drop=True)
worklist_df["new_rank"] = np.arange(1, len(worklist_df) + 1)

worklist_df[[
    "image_id",
    "original_rank",
    "new_rank",
    "top_finding",
    "top_probability",
    "urgency_score",
    "triage_tier"
]]

In [ ]:
worklist_df.to_csv(WORKLIST_PATH, index=False)
print("Saved worklist simulation to:", WORKLIST_PATH)

In [ ]:
top_case_path = worklist_df.iloc[0]["image_path"]
gray_img2, rgb_img2, tensor2, probs2, result_df2 = predict_single_image(model, top_case_path, best_thresholds)

plt.figure(figsize=(5, 5))
plt.imshow(gray_img2, cmap="gray")
plt.axis("off")
plt.show()

print("Top finding:", result_df2.iloc[0]["class"])
print("Top probability:", float(result_df2.iloc[0]["probability"]))
print("Urgency score:", compute_triage_score(probs2))
print("Triage tier:", assign_triage_tier(probs2, compute_triage_score(probs2)))

result_df2.head(10)

In [ ]:
try:
    gradcam.close()
except:
    pass

gc.collect()
torch.cuda.empty_cache()
print("Cleanup done.")